In [32]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

from transaction_analysis.models.features import build_features

X, y = build_features(model_training=True)

X, y = resample(X, y, n_samples=2_000_000, stratify=y, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [34]:
from transaction_analysis.models.config import RFConfig, XGBConfig
from transaction_analysis.models.factory import ModelType, train_model

rf_model = train_model(ModelType.RF, RFConfig(max_depth=5, n_estimators=100), X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:   10.6s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   33.7s finished


In [35]:
xgb_model = train_model(ModelType.XGB, XGBConfig(max_depth=5, n_estimators=100), X_train, y_train)

/mnt/shared/courses/1/analiza-danych/TransactionDatasetAnalysis/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [15:17:57] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_threads", "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [36]:
from transaction_analysis.models.factory import evaluate_model

results_xgb = evaluate_model(xgb_model, X_test, y_test)
results_rf = evaluate_model(rf_model, X_test, y_test)

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.2s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.3s finished


In [37]:
from sklearn.metrics import classification_report

print("Random Forest Classification Report:")
print(classification_report(results_rf["y_test"], results_rf["y_pred"]))

print("XGBoost Classification Report:")
print(classification_report(results_xgb["y_test"], results_xgb["y_pred"]))

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.85      0.92    399421
           1       0.01      0.75      0.01       579

    accuracy                           0.85    400000
   macro avg       0.50      0.80      0.47    400000
weighted avg       1.00      0.85      0.92    400000

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    399421
           1       0.54      0.42      0.47       579

    accuracy                           1.00    400000
   macro avg       0.77      0.71      0.74    400000
weighted avg       1.00      1.00      1.00    400000

